In [25]:
import os
import json
import pdfplumber
from openai import OpenAI


client = OpenAI(
    base_url="http://localhost:11434/v1", 
    api_key="ollama"  
)

In [26]:

SYSTEM_PROMPT = """You are a precise document data extractor.

Your task is to extract specific information from the invoice text provided by the user.

Return your response as a valid JSON object with EXACTLY these fields:
{
  "invoice_number": "The unique invoice ID or reference number",
  "invoice_date": "Date the invoice was issued — format DD-MM-YYYY",
  "due_date": "Payment deadline — null if not mentioned",
  "billed_by": "Name of the company or person issuing the invoice",
  "billed_to": "Name of the client or recipient being billed",
  "line_items": [{"item": "service or product name", "amount": "cost of item"}],
  "subtotal": "Total before any discounts or taxes",
  "discount": "Any discount applied — null if none",
  "tax_or_gst": "Tax amount if mentioned — null if not applicable",
  "total_amount": "The final payable amount",
  "currency": "Currency used in the invoice (e.g., INR, USD)",
  "payment_method": "How payment should be made — null if not mentioned",
  "notes": "Any additional remarks or instructions on the invoice"
}

Rules:
- Return ONLY the raw JSON object. Do NOT wrap it in markdown code fences (like ```json). No explanation.
- If a field cannot be found in the document, set its value to null. Do not use "N/A" or empty strings.
- Do not guess or make up values. Only extract what is clearly present in the text.
"""

In [27]:
def extract_text_from_pdf(pdf_path):
    
    raw_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                raw_text += page.extract_text() + "\n"
    return raw_text

def clean_extracted_text(text):
    
    lines = text.split("\n")
    cleaned_lines = [line.strip() for line in lines if line.strip()]
    return "\n".join(cleaned_lines)

In [28]:
def extract_invoice_data(cleaned_text):
    response = client.chat.completions.create(
        model="llama3.2:1b",  
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract from this document:\n\n{cleaned_text}"}
        ],
        temperature=0  
    )
    return response.choices[0].message.content

In [29]:
def process_pipeline(pdf_path, output_json_path):
    
    print(f"\n--- Processing: {pdf_path} ---")
    
    # 1. Extract
    raw_text = extract_text_from_pdf(pdf_path)
    if not raw_text.strip():
        print(f"Error: No text could be extracted from {pdf_path}.")
        return

    # 2. Clean
    cleaned_text = clean_extracted_text(raw_text)
    
    # 3. Call LLM
    raw_llm_response = extract_invoice_data(cleaned_text)
    
    # 4. Parse JSON 
    try:
        json_data = json.loads(raw_llm_response)
        
        # 5. Display and Save Output
        print("Extraction Successful! Sample Output:")
        print(json.dumps(json_data, indent=2, ensure_ascii=False))
        
        # Ensure outputs/ folder exists
        os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
        
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)
        print(f"Saved to: {output_json_path}")
        
    except json.JSONDecodeError:
        print("JSON Parsing Failed. Raw LLM response received:")
        print(raw_llm_response)

In [30]:
invoice_mappings = [
    ("invoices/invoice-1.pdf", "outputs/output_invoice_1.json"),
    ("invoices/invoice-2.pdf", "outputs/output_invoice_2.json"),
    ("invoices/invoice-3.pdf", "outputs/output_invoice_3.json"),
]

for pdf_in, json_out in invoice_mappings:
    if os.path.exists(pdf_in):
        process_pipeline(pdf_in, json_out)
    else:
        print(f"Warning: Put your invoice file at '{pdf_in}' to run the pipeline.")


--- Processing: invoices/invoice-1.pdf ---
Extraction Successful! Sample Output:
{
  "invoice_number": "INV-3337",
  "invoice_date": "01-01-2016",
  "due_date": "31-01-2016",
  "billed_by": "admin@slicedinvoices.com",
  "billed_to": "Test Business",
  "line_items": [
    {
      "item": "Web Design",
      "amount": "1.00"
    }
  ],
  "subtotal": "85.00",
  "discount": null,
  "tax_or_gst": "8.50",
  "total_amount": "93.50",
  "currency": "AUD",
  "payment_method": "ANZ Bank",
  "notes": null
}
Saved to: outputs/output_invoice_1.json

--- Processing: invoices/invoice-2.pdf ---
Extraction Successful! Sample Output:
{
  "invoice_number": "1213",
  "invoice_date": "16.12.2021",
  "due_date": null,
  "billed_by": "La Galerie",
  "billed_to": "Same as recipient",
  "line_items": [
    {
      "item": "Glossostigma",
      "amount": "29.70"
    },
    {
      "item": "Bayberry",
      "amount": "1,207.68"
    },
    {
      "item": "Waxflower",
      "amount": "56.78"
    },
    {
      "i